In [1]:
 !pip install scikit-image


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# SOFT MCS BUILDING WITH SCIKIT-IMAGE
#
# Method A:
#   Merge fragmented cold-cloud blobs within one frame using
#   binary closing + binary dilation before connected-component labeling.
#
# Method B:
#   Softer tracking between frames using:
#     1. original-mask IoU
#     2. buffered-mask IoU
#     3. centroid distance
#   with a short time-window link.
# ============================================================

In [15]:
# ============================================================
# SOFT MCS BUILDING — SCIPY VERSION
#
# Uses existing WNP MERGIR files:
#   merg_YYYYMMDDHH_4km-pixel_WNP.nc4
#
# Method A:
#   binary closing + binary dilation before labeling
#
# Method B:
#   soft tracking using original-mask IoU, buffered-mask IoU,
#   centroid distance, and short link window
# ============================================================

import os
import glob
import calendar
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from scipy.ndimage import (
    label,
    binary_closing,
    binary_dilation,
    generate_binary_structure
)

# ============================================================
# CONFIG
# ============================================================
CFG = {
    # Existing cropped WNP MERGIR files
    "WNP_DIR": Path(r"C:\Users\_s2218026\Documents\lab\mergir_wnp"),

    # Existing SciPy-soft output folder
    "OUT_DIR": Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest"),

    # WNP domain
    "LAT_MIN": 0.0,
    "LAT_MAX": 30.0,
    "LON_MIN": 100.0,
    "LON_MAX": 180.0,

    # MCS detection
    "TB_TH": 233.0,
    "AREA_MIN_KM2": 5000.0,

    # Method A: within-frame soft merge
    "USE_CLOSING": True,
    "CLOSE_ITERS": 1,
    "MERGE_ITERS": 2,

    # Method B: between-frame soft tracking
    "TRACK_BUFFER_ITERS": 2,
    "TRACK_MIN_IOU_ORIG": 0.01,
    "TRACK_MIN_IOU_BUF": 0.01,
    "TRACK_MAX_KM": 300.0,
    "TRACK_LINK_WINDOW_H": 6.0,
}

CFG["OUT_DIR"].mkdir(parents=True, exist_ok=True)

# 8-connected 2D structure
STRUCT2D = generate_binary_structure(2, 2)

YEARS_FULL = list(range(2009, 2026))
YEARS_RESUME = list(range(2018, 2026))
MONTHS = [6, 7, 8, 9, 10]

print("WNP_DIR:", CFG["WNP_DIR"])
print("OUT_DIR:", CFG["OUT_DIR"])

WNP_DIR: C:\Users\_s2218026\Documents\lab\mergir_wnp
OUT_DIR: C:\Users\_s2218026\Documents\lab\mcs_output_softtest


In [16]:
# ============================================================
# GENERAL HELPERS
# ============================================================

def pixel_area_km2(lat_vals, lon_vals):
    """
    Approximate pixel area on regular lat-lon grid.
    Returns area per latitude row.
    """
    R = 6371.0

    lat_vals = np.asarray(lat_vals, dtype=float)
    lon_vals = np.asarray(lon_vals, dtype=float)

    lat = np.deg2rad(lat_vals)

    dlat = np.abs(np.deg2rad(lat_vals[1] - lat_vals[0]))
    dlon = np.abs(np.deg2rad(lon_vals[1] - lon_vals[0]))

    return (R ** 2) * dlat * dlon * np.cos(lat)


def linset_to_mask(linset, shape):
    mask = np.zeros(shape, dtype=bool)

    if len(linset) == 0:
        return mask

    arr = np.fromiter(linset, dtype=np.int64)
    yy = arr // shape[1]
    xx = arr % shape[1]

    mask[yy, xx] = True

    return mask


def mask_to_linset(mask):
    yy, xx = np.where(mask)
    return set((yy.astype(np.int64) * mask.shape[1] + xx.astype(np.int64)).tolist())


def dilate_mask_iters(mask, n_iter):
    if n_iter <= 0:
        return mask.copy()

    out = mask.copy()

    for _ in range(n_iter):
        out = binary_dilation(out, structure=STRUCT2D)

    return out


def close_mask_iters(mask, n_iter):
    if n_iter <= 0:
        return mask.copy()

    out = mask.copy()

    for _ in range(n_iter):
        out = binary_closing(out, structure=STRUCT2D)

    return out


def dilate_linset(linset, shape, n_iter):
    mask = linset_to_mask(linset, shape)
    buf = dilate_mask_iters(mask, n_iter)
    return mask_to_linset(buf)


def compute_iou(set_a, set_b):
    if not set_a or not set_b:
        return 0.0

    inter = len(set_a & set_b)

    if inter == 0:
        return 0.0

    union = len(set_a | set_b)

    return inter / union


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.asarray(lat1, dtype=float)
    lon1 = np.asarray(lon1, dtype=float)
    lat2 = np.asarray(lat2, dtype=float)
    lon2 = np.asarray(lon2, dtype=float)

    lat1_rad = np.deg2rad(lat1)
    lon1_rad = np.deg2rad(lon1)
    lat2_rad = np.deg2rad(lat2)
    lon2_rad = np.deg2rad(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0) ** 2
    )

    return 2.0 * R * np.arcsin(np.sqrt(a))


def bearing_deg(lat1, lon1, lat2, lon2):
    lat1 = np.asarray(lat1, dtype=float)
    lon1 = np.asarray(lon1, dtype=float)
    lat2 = np.asarray(lat2, dtype=float)
    lon2 = np.asarray(lon2, dtype=float)

    lat1_rad = np.deg2rad(lat1)
    lon1_rad = np.deg2rad(lon1)
    lat2_rad = np.deg2rad(lat2)
    lon2_rad = np.deg2rad(lon2)

    dlon = lon2_rad - lon1_rad

    x = np.sin(dlon) * np.cos(lat2_rad)
    y = (
        np.cos(lat1_rad) * np.sin(lat2_rad)
        - np.sin(lat1_rad) * np.cos(lat2_rad) * np.cos(dlon)
    )

    brng = np.degrees(np.arctan2(x, y))

    return (brng + 360.0) % 360.0

In [17]:
# ============================================================
# OPEN WNP MERGIR DAILY DATA
# ============================================================

def snap_time_to_30min(ds):
    t = pd.to_datetime(ds["time"].values).round("30min")
    return ds.assign_coords(time=("time", t))


def open_day_wnp_30min(date_str, cfg=CFG):
    """
    Opens 24 hourly WNP-cropped MERGIR files for one date.
    Expected files:
      merg_YYYYMMDDHH_4km-pixel_WNP.nc4
    """
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")

    files = sorted(
        glob.glob(str(cfg["WNP_DIR"] / f"merg_{ymd}??_4km-pixel_WNP.nc4"))
    )

    if len(files) != 24:
        raise FileNotFoundError(
            f"Need 24 WNP hourly files for {ymd}; found {len(files)}."
        )

    dsets = []

    for fp in files:
        ds = xr.open_dataset(fp, engine="netcdf4")[["Tb"]]
        ds = snap_time_to_30min(ds)
        dsets.append(ds)

    day = xr.concat(dsets, dim="time").sortby("time")

    # remove duplicate timestamps if any
    t = pd.to_datetime(day.time.values)
    _, idx = np.unique(t.values, return_index=True)
    day = day.isel(time=np.sort(idx))

    return day


def make_day_3hourly(date_str, cfg=CFG):
    day_30m = open_day_wnp_30min(date_str, cfg)
    day_3h = day_30m.resample(time="3h").nearest()
    return day_30m, day_3h

In [18]:
# ============================================================
# METHOD A: SOFT OBJECT DETECTION WITH SCIPY
# ============================================================

def detect_objects_one_time_soft(
    Tb2d,
    lat_vals,
    lon_vals,
    tb_th,
    area_min_km2,
    use_closing=True,
    close_iters=1,
    merge_iters=2,
    track_buffer_iters=2,
):
    """
    Soft object detection logic:

    1. Build original cold cloud mask: Tb < threshold.
    2. Apply binary closing to fill small gaps.
    3. Apply binary dilation to merge nearby fragments.
    4. Label connected components using scipy.ndimage.label.
    5. Compute object statistics using ORIGINAL cold pixels only.
    6. Save original and buffered pixel sets for soft tracking.
    """
    orig_mask = np.isfinite(Tb2d) & (Tb2d < tb_th)

    work_mask = orig_mask.copy()

    if use_closing and close_iters > 0:
        work_mask = close_mask_iters(work_mask, close_iters)

    if merge_iters > 0:
        work_mask = dilate_mask_iters(work_mask, merge_iters)

    lab, nlab = label(work_mask, structure=STRUCT2D)

    if nlab == 0:
        return []

    arow = pixel_area_km2(lat_vals, lon_vals)
    nlon = len(lon_vals)
    shape = Tb2d.shape

    objs = []

    for oid in range(1, nlab + 1):
        merged_region = lab == oid

        # Use original cold pixels inside the merged region.
        # This preserves the physical Tb < threshold definition.
        member_mask = orig_mask & merged_region

        yy, xx = np.where(member_mask)

        if len(yy) == 0:
            continue

        area = float(arow[yy].sum())

        if area < area_min_km2:
            continue

        tb = Tb2d[yy, xx]
        w = arow[yy]

        clat = float(np.sum(lat_vals[yy] * w) / np.sum(w))
        clon = float(np.sum(lon_vals[xx] * w) / np.sum(w))

        lin_orig = set((yy.astype(np.int64) * nlon + xx.astype(np.int64)).tolist())
        lin_buf = dilate_linset(lin_orig, shape, track_buffer_iters)

        objs.append({
            "area_km2": area,
            "centroid_lat": clat,
            "centroid_lon": clon,
            "tb_mean": float(np.nanmean(tb)),
            "tb_min": float(np.nanmin(tb)),
            "npix": int(len(yy)),
            "linpix_orig": lin_orig,
            "linpix_buf": lin_buf,
        })

    return objs


def detect_objects_day_soft(day_3h, cfg=CFG):
    lat_vals = day_3h["lat"].values
    lon_vals = day_3h["lon"].values

    rows = []
    pix_index = {}

    for tt in day_3h.time.values:
        tstamp = pd.Timestamp(tt)
        Tb2d = day_3h["Tb"].sel(time=tt).values

        objs = detect_objects_one_time_soft(
            Tb2d=Tb2d,
            lat_vals=lat_vals,
            lon_vals=lon_vals,
            tb_th=cfg["TB_TH"],
            area_min_km2=cfg["AREA_MIN_KM2"],
            use_closing=cfg["USE_CLOSING"],
            close_iters=cfg["CLOSE_ITERS"],
            merge_iters=cfg["MERGE_ITERS"],
            track_buffer_iters=cfg["TRACK_BUFFER_ITERS"],
        )

        for j, ob in enumerate(objs):
            obj_key = f"{tstamp:%Y%m%d%H%M}_{j:04d}"

            pix_index[(tstamp, obj_key)] = {
                "orig": ob["linpix_orig"],
                "buf": ob["linpix_buf"],
            }

            rows.append({
                "time": tstamp,
                "obj_key": obj_key,
                "area_km2": ob["area_km2"],
                "centroid_lat": ob["centroid_lat"],
                "centroid_lon": ob["centroid_lon"],
                "tb_mean": ob["tb_mean"],
                "tb_min": ob["tb_min"],
                "npix": ob["npix"],
            })

    if len(rows) == 0:
        return pd.DataFrame(
            columns=[
                "time", "obj_key", "area_km2", "centroid_lat",
                "centroid_lon", "tb_mean", "tb_min", "npix"
            ]
        ), pix_index

    objects_df = (
        pd.DataFrame(rows)
        .sort_values(["time", "obj_key"])
        .reset_index(drop=True)
    )

    return objects_df, pix_index

In [19]:
# ============================================================
# METHOD B: SOFT TRACKING
# ============================================================

def track_objects_soft(
    objects_df,
    pix_index,
    min_iou_orig=0.01,
    min_iou_buf=0.01,
    max_km=300.0,
    link_window_h=6.0,
):
    """
    Soft tracking logic:
      1. Try original-mask IoU.
      2. Try buffered-mask IoU.
      3. Try centroid distance.
      4. Allow short missing-frame gap using link_window_h.
    """
    if objects_df.empty:
        out = objects_df.copy()
        out["track_id"] = pd.Series(dtype="Int64")
        out["dt_h"] = np.nan
        out["dist_km"] = np.nan
        out["speed_kmh"] = np.nan
        out["direction_deg"] = np.nan
        return out

    df = objects_df.copy()
    df["track_id"] = None

    times = sorted(df["time"].unique())

    next_tid = 1
    active = {}

    for tb in times:
        tb = pd.Timestamp(tb)

        B = df[df["time"] == tb].sort_values("area_km2", ascending=False)
        used_tids_this_time = set()

        # Remove stale active tracks
        active = {
            tid: info
            for tid, info in active.items()
            if ((tb - pd.Timestamp(info["time"])) / pd.Timedelta(hours=1)) <= link_window_h
        }

        for b_idx in B.index:
            b = df.loc[b_idx]

            bpix_orig = pix_index[(tb, b["obj_key"])]["orig"]
            bpix_buf = pix_index[(tb, b["obj_key"])]["buf"]

            candidates = []

            for tid, info in active.items():
                if tid in used_tids_this_time:
                    continue

                a_idx = info["idx"]
                ta = pd.Timestamp(info["time"])

                dt_h = (tb - ta) / pd.Timedelta(hours=1)

                if dt_h <= 0 or dt_h > link_window_h:
                    continue

                a = df.loc[a_idx]

                apix_orig = pix_index[(pd.Timestamp(a["time"]), a["obj_key"])]["orig"]
                apix_buf = pix_index[(pd.Timestamp(a["time"]), a["obj_key"])]["buf"]

                iou_orig = compute_iou(apix_orig, bpix_orig)
                iou_buf = compute_iou(apix_buf, bpix_buf)

                dist_km = float(
                    haversine_km(
                        a["centroid_lat"],
                        a["centroid_lon"],
                        b["centroid_lat"],
                        b["centroid_lon"],
                    )
                )

                candidates.append({
                    "tid": tid,
                    "iou_orig": iou_orig,
                    "iou_buf": iou_buf,
                    "dist_km": dist_km,
                    "dt_h": float(dt_h),
                })

            assigned_tid = None

            # Priority 1: original overlap
            orig_ok = [c for c in candidates if c["iou_orig"] >= min_iou_orig]

            if orig_ok:
                best = sorted(
                    orig_ok,
                    key=lambda c: (-c["iou_orig"], c["dist_km"], c["dt_h"])
                )[0]
                assigned_tid = best["tid"]

            # Priority 2: buffered overlap
            if assigned_tid is None:
                buf_ok = [c for c in candidates if c["iou_buf"] >= min_iou_buf]

                if buf_ok:
                    best = sorted(
                        buf_ok,
                        key=lambda c: (-c["iou_buf"], c["dist_km"], c["dt_h"])
                    )[0]
                    assigned_tid = best["tid"]

            # Priority 3: centroid distance
            if assigned_tid is None:
                dist_ok = [c for c in candidates if c["dist_km"] <= max_km]

                if dist_ok:
                    best = sorted(
                        dist_ok,
                        key=lambda c: (c["dist_km"], c["dt_h"])
                    )[0]
                    assigned_tid = best["tid"]

            # Create new track if no match
            if assigned_tid is None:
                assigned_tid = next_tid
                next_tid += 1

            df.loc[b_idx, "track_id"] = assigned_tid

            active[assigned_tid] = {
                "idx": b_idx,
                "time": tb,
            }

            used_tids_this_time.add(assigned_tid)

    df["track_id"] = df["track_id"].astype(int)

    df = df.sort_values(["track_id", "time"]).reset_index(drop=True)

    lat_prev = df.groupby("track_id")["centroid_lat"].shift(1)
    lon_prev = df.groupby("track_id")["centroid_lon"].shift(1)
    t_prev = df.groupby("track_id")["time"].shift(1)

    df["dt_h"] = (df["time"] - t_prev).dt.total_seconds() / 3600.0

    df["dist_km"] = haversine_km(
        lat_prev.values,
        lon_prev.values,
        df["centroid_lat"].values,
        df["centroid_lon"].values,
    )

    df["speed_kmh"] = df["dist_km"] / df["dt_h"]

    df["direction_deg"] = bearing_deg(
        lat_prev.values,
        lon_prev.values,
        df["centroid_lat"].values,
        df["centroid_lon"].values,
    )

    return df

In [20]:
# ============================================================
# ONE-DAY SOFT RUNNER
# ============================================================

def run_one_day_soft(date_str, cfg=CFG):
    print(f"\n=== RUN SOFT SCIPY DAY: {date_str} ===")

    day_30m = None
    day_3h = None

    try:
        day_30m, day_3h = make_day_3hourly(date_str, cfg)

        print("30-min ntimes:", day_30m.sizes["time"])
        print("3-hour ntimes:", day_3h.sizes["time"])

        objects_df, pix_index = detect_objects_day_soft(day_3h, cfg)

        tracks_df = track_objects_soft(
            objects_df=objects_df,
            pix_index=pix_index,
            min_iou_orig=cfg["TRACK_MIN_IOU_ORIG"],
            min_iou_buf=cfg["TRACK_MIN_IOU_BUF"],
            max_km=cfg["TRACK_MAX_KM"],
            link_window_h=cfg["TRACK_LINK_WINDOW_H"],
        )

        ymd = pd.Timestamp(date_str).strftime("%Y%m%d")

        objects_csv = cfg["OUT_DIR"] / f"mcs_objects_soft_3h_{ymd}.csv"
        tracks_csv = cfg["OUT_DIR"] / f"mcs_tracks_soft_3h_{ymd}.csv"

        objects_df.to_csv(objects_csv, index=False)
        tracks_df.to_csv(tracks_csv, index=False)

        print("[SAVED]", objects_csv.name, "| rows:", len(objects_df))
        print("[SAVED]", tracks_csv.name, "| rows:", len(tracks_df))

        if "track_id" in tracks_df.columns:
            print("n tracks:", tracks_df["track_id"].nunique())

        return objects_df, tracks_df

    finally:
        try:
            if day_30m is not None:
                day_30m.close()
        except Exception:
            pass

        try:
            if day_3h is not None:
                day_3h.close()
        except Exception:
            pass

In [21]:
# ============================================================
# MONTHLY CONCAT + RESUME RUNNER
# ============================================================

def concat_month_soft_safe(year, month, cfg=CFG):
    ym = f"{year}{month:02d}"

    out_tracks = cfg["OUT_DIR"] / f"mcs_tracks_soft_3h_{ym}.csv"
    out_objects = cfg["OUT_DIR"] / f"mcs_objects_soft_3h_{ym}.csv"

    track_files = sorted(
        glob.glob(str(cfg["OUT_DIR"] / f"mcs_tracks_soft_3h_{ym}??.csv"))
    )

    object_files = sorted(
        glob.glob(str(cfg["OUT_DIR"] / f"mcs_objects_soft_3h_{ym}??.csv"))
    )

    expected_days = calendar.monthrange(year, month)[1]

    if len(track_files) != expected_days or len(object_files) != expected_days:
        raise FileNotFoundError(
            f"Incomplete daily SOFT files for {ym}. "
            f"Expected {expected_days}, got tracks={len(track_files)}, objects={len(object_files)}"
        )

    tracks = pd.concat([pd.read_csv(f) for f in track_files], ignore_index=True)
    objs = pd.concat([pd.read_csv(f) for f in object_files], ignore_index=True)

    tracks["time"] = pd.to_datetime(tracks["time"], errors="coerce")
    objs["time"] = pd.to_datetime(objs["time"], errors="coerce")

    tracks["date"] = tracks["time"].dt.strftime("%Y%m%d")
    tracks["track_id_month"] = (
        tracks["date"].astype(str)
        + "_"
        + tracks["track_id"].astype(int).astype(str)
    )

    tracks.to_csv(out_tracks, index=False)
    objs.to_csv(out_objects, index=False)

    print(
        f"[MONTH SAVED] {out_tracks.name} "
        f"rows={len(tracks)} unique_tracks={tracks['track_id_month'].nunique()}"
    )

    print(f"[MONTH SAVED] {out_objects.name} rows={len(objs)}")

    return out_tracks, out_objects


def run_one_month_soft_safe(year, month, cfg=CFG, skip_existing=True):
    ndays = calendar.monthrange(year, month)[1]

    print("\n" + "=" * 70)
    print(f"SOFT SCIPY MONTH RUN: {year}-{month:02d} | days={ndays}")
    print("=" * 70)

    for day in range(1, ndays + 1):
        ds = f"{year}-{month:02d}-{day:02d}"
        ymd = f"{year}{month:02d}{day:02d}"

        out_trk = cfg["OUT_DIR"] / f"mcs_tracks_soft_3h_{ymd}.csv"
        out_obj = cfg["OUT_DIR"] / f"mcs_objects_soft_3h_{ymd}.csv"

        if (
            skip_existing
            and out_trk.exists()
            and out_obj.exists()
            and out_trk.stat().st_size > 1000
            and out_obj.stat().st_size > 1000
        ):
            print(f"[{day:02d}/{ndays}] SKIP {ds}")
            continue

        print(f"[{day:02d}/{ndays}] RUN {ds}")

        try:
            run_one_day_soft(ds, cfg)
            print(f"[{day:02d}/{ndays}] DONE {ds}")

        except Exception as e:
            print(f"[{day:02d}/{ndays}] FAILED {ds} -> {e}")

    print(f"\n=== CONCAT SOFT SCIPY MONTH {year}-{month:02d} ===")
    concat_month_soft_safe(year, month, cfg)
    print(f"=== SOFT SCIPY MONTH FINISHED {year}-{month:02d} ===")


def run_soft_scipy_resume_2018_2025(cfg=CFG, skip_existing=True):
    for y in range(2018, 2026):
        for m in [6, 7, 8, 9, 10]:
            run_one_month_soft_safe(
                year=y,
                month=m,
                cfg=cfg,
                skip_existing=skip_existing
            )

In [22]:
# ============================================================
# FAST WNP AUDIT FOR 2018–2025 JJASO
# ============================================================

from collections import Counter

def fast_wnp_audit(years=YEARS_RESUME, months=MONTHS, cfg=CFG):
    WNP_DIR = cfg["WNP_DIR"]

    print("Scanning WNP folder once...")
    files = list(WNP_DIR.glob("merg_*_4km-pixel_WNP.nc4"))
    print("Total WNP files found:", len(files))

    ymdh_list = []

    for fp in files:
        try:
            ymdh = fp.name.split("_")[1]
            if len(ymdh) == 10:
                ymdh_list.append(ymdh)
        except Exception:
            pass

    day_counts = Counter([x[:8] for x in ymdh_list])

    missing_days = []

    for y in years:
        for m in months:
            ndays = calendar.monthrange(y, m)[1]

            for d in range(1, ndays + 1):
                ymd = f"{y}{m:02d}{d:02d}"
                n = day_counts.get(ymd, 0)

                if n != 24:
                    missing_days.append({
                        "date": f"{y}-{m:02d}-{d:02d}",
                        "ymd": ymd,
                        "found_hours": n,
                    })

    missing_df = pd.DataFrame(missing_days)

    print("\nExpected days:", sum(calendar.monthrange(y, m)[1] for y in years for m in months))
    print("Missing/incomplete days:", len(missing_df))

    if len(missing_df) > 0:
        display(missing_df)
    else:
        print("All WNP hourly files are complete.")

    return missing_df


missing_wnp = fast_wnp_audit()

Scanning WNP folder once...
Total WNP files found: 62424

Expected days: 1224
Missing/incomplete days: 0
All WNP hourly files are complete.


In [ ]:
# ============================================================
# RUN SCIPY-SOFT RESUME: 2018–2025
# ============================================================

run_soft_scipy_resume_2018_2025(
    cfg=CFG,
    skip_existing=True
)


SOFT SCIPY MONTH RUN: 2018-06 | days=30
[01/30] RUN 2018-06-01

=== RUN SOFT SCIPY DAY: 2018-06-01 ===
30-min ntimes: 48
3-hour ntimes: 8
[SAVED] mcs_objects_soft_3h_20180601.csv | rows: 278
[SAVED] mcs_tracks_soft_3h_20180601.csv | rows: 278
n tracks: 111
[01/30] DONE 2018-06-01
[02/30] RUN 2018-06-02

=== RUN SOFT SCIPY DAY: 2018-06-02 ===
30-min ntimes: 48
3-hour ntimes: 8
[SAVED] mcs_objects_soft_3h_20180602.csv | rows: 324
[SAVED] mcs_tracks_soft_3h_20180602.csv | rows: 324
n tracks: 127
[02/30] DONE 2018-06-02
[03/30] RUN 2018-06-03

=== RUN SOFT SCIPY DAY: 2018-06-03 ===
30-min ntimes: 48
3-hour ntimes: 8
[SAVED] mcs_objects_soft_3h_20180603.csv | rows: 384
[SAVED] mcs_tracks_soft_3h_20180603.csv | rows: 384
n tracks: 138
[03/30] DONE 2018-06-03
[04/30] RUN 2018-06-04

=== RUN SOFT SCIPY DAY: 2018-06-04 ===
30-min ntimes: 48
3-hour ntimes: 8
[SAVED] mcs_objects_soft_3h_20180604.csv | rows: 289
[SAVED] mcs_tracks_soft_3h_20180604.csv | rows: 289
n tracks: 120
[04/30] DONE 2018-0

In [ ]:
# ============================================================
# AUDIT FULL SOFT SCIPY MONTHLY FILES: 2009–2025
# ============================================================

def audit_soft_monthly_full(years=YEARS_FULL, months=MONTHS, cfg=CFG):
    expected = [
        cfg["OUT_DIR"] / f"mcs_tracks_soft_3h_{y}{m:02d}.csv"
        for y in years
        for m in months
    ]

    missing = [fp.name for fp in expected if not fp.exists()]

    print("Expected monthly SOFT files:", len(expected))
    print("Missing:", len(missing))

    if missing:
        print("\nMissing files:")
        for f in missing:
            print(f)

    qc_rows = []

    for fp in expected:
        if not fp.exists():
            continue

        df = pd.read_csv(fp, usecols=["time", "track_id"])
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

        qc_rows.append({
            "file": fp.name,
            "rows": len(df),
            "n_tracks": df["track_id"].nunique(),
            "time_min": df["time"].min(),
            "time_max": df["time"].max(),
        })

    qc = pd.DataFrame(qc_rows)

    if len(qc) > 0:
        display(qc.tail(20))

        print("\nOverall:")
        print("Files:", len(qc))
        print("Rows:", qc["rows"].sum())
        print("Min:", qc["time_min"].min())
        print("Max:", qc["time_max"].max())

    return qc


qc_soft_monthly = audit_soft_monthly_full()

In [ ]:
# ============================================================
# CONCATENATE FINAL SOFT SCIPY 2009–2025 JJASO PERIOD FILE
# ============================================================

def concat_final_soft_period(years=YEARS_FULL, months=MONTHS, cfg=CFG):
    out_csv = cfg["OUT_DIR"] / "mcs_tracks_soft_3h_2009_2025_JJASO.csv"

    files = [
        cfg["OUT_DIR"] / f"mcs_tracks_soft_3h_{y}{m:02d}.csv"
        for y in years
        for m in months
    ]

    missing = [fp for fp in files if not fp.exists()]

    if missing:
        raise FileNotFoundError(
            "Missing SOFT monthly files:\n"
            + "\n".join(str(fp) for fp in missing)
        )

    dfs = []

    for fp in files:
        print("reading", fp.name)
        df = pd.read_csv(fp)
        df["source_file"] = fp.name
        dfs.append(df)

    soft = pd.concat(dfs, ignore_index=True)

    soft["time"] = (
        pd.to_datetime(soft["time"], errors="coerce", utc=True)
        .dt.tz_convert(None)
    )

    soft = soft.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()

    soft = soft[
        soft["time"].dt.year.between(2009, 2025)
        & soft["time"].dt.month.isin(months)
    ].copy()

    # Preserve original daily/monthly soft track IDs
    soft["date"] = soft["time"].dt.strftime("%Y%m%d")
    soft["track_id_original"] = soft["track_id"]

    # Globally unique soft track ID
    soft["track_id"] = (
        "soft_"
        + soft["date"].astype(str)
        + "_"
        + soft["track_id_original"].astype(str)
    )

    soft = soft.sort_values(["time", "track_id"]).reset_index(drop=True)

    soft.to_csv(out_csv, index=False)

    print("\n[SAVED FINAL SOFT PERIOD FILE]")
    print(out_csv)

    print("\nQC:")
    print("Rows:", len(soft))
    print("Unique tracks:", soft["track_id"].nunique())
    print("Time range:", soft["time"].min(), "to", soft["time"].max())
    print("Years:", soft["time"].dt.year.min(), "-", soft["time"].dt.year.max())
    print("Months:", sorted(soft["time"].dt.month.unique()))

    return out_csv, soft


SOFT_PERIOD_CSV, soft_period = concat_final_soft_period()

In [ ]:
# ============================================================
# QC COMPARISON: OLD VS SOFT FOR ONE DAY
# ============================================================

def load_day_outputs(date_str):
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")

    old_dir = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output")
    soft_dir = CFG["OUT_DIR"]

    old_obj_fp = old_dir / f"mcs_objects_3h_{ymd}.csv"
    old_trk_fp = old_dir / f"mcs_tracks_3h_{ymd}.csv"

    soft_obj_fp = soft_dir / f"mcs_objects_soft_3h_{ymd}.csv"
    soft_trk_fp = soft_dir / f"mcs_tracks_soft_3h_{ymd}.csv"

    old_obj = pd.read_csv(old_obj_fp)
    old_trk = pd.read_csv(old_trk_fp)
    soft_obj = pd.read_csv(soft_obj_fp)
    soft_trk = pd.read_csv(soft_trk_fp)

    for df in [old_obj, old_trk, soft_obj, soft_trk]:
        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce")

    return old_obj, old_trk, soft_obj, soft_trk


def summarize_catalog(obj_df, trk_df, name="dataset"):
    out = {
        "dataset": name,
        "n_objects": len(obj_df),
        "n_track_rows": len(trk_df),
        "n_tracks": trk_df["track_id"].nunique() if "track_id" in trk_df.columns else np.nan,
        "mean_area_km2": obj_df["area_km2"].mean() if "area_km2" in obj_df.columns else np.nan,
        "median_area_km2": obj_df["area_km2"].median() if "area_km2" in obj_df.columns else np.nan,
        "mean_tb_mean": obj_df["tb_mean"].mean() if "tb_mean" in obj_df.columns else np.nan,
        "mean_tb_min": obj_df["tb_min"].mean() if "tb_min" in obj_df.columns else np.nan,
        "mean_speed_kmh": trk_df["speed_kmh"].dropna().mean() if "speed_kmh" in trk_df.columns else np.nan,
        "median_speed_kmh": trk_df["speed_kmh"].dropna().median() if "speed_kmh" in trk_df.columns else np.nan,
    }

    return pd.DataFrame([out])


def qc_summary_table(date_str):
    old_obj, old_trk, soft_obj, soft_trk = load_day_outputs(date_str)

    summary = pd.concat(
        [
            summarize_catalog(old_obj, old_trk, "old"),
            summarize_catalog(soft_obj, soft_trk, "soft_scikit_image"),
        ],
        ignore_index=True
    )

    return summary, old_obj, old_trk, soft_obj, soft_trk


summary, old_obj, old_trk, soft_obj, soft_trk = qc_summary_table("2015-08-20")
summary